In [1]:
import pytesseract #obraz na tekst
from PIL import Image
import re
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import PatternFill
from datetime import datetime
import os
import numpy as np
!pip show pytesseract
!where python

Name: pytesseract
Version: 0.3.13
Summary: Python-tesseract is a python wrapper for Google's Tesseract-OCR
Home-page: https://github.com/madmaze/pytesseract
Author: Samuel Hoffstaetter
Author-email: samuel@hoffstaetter.com
License: Apache License 2.0
Location: C:\Users\Yoga\AppData\Local\Programs\Python\Python312\Lib\site-packages
Requires: packaging, Pillow
Required-by: 
c:\Users\Yoga\AppData\Local\Programs\Python\Python312\python.exe
C:\Users\Yoga\AppData\Local\Microsoft\WindowsApps\python.exe


In [2]:
def get_exam_date():
    user_date = input("📅 Podaj datę badania [RRRR-MM-DD] (Enter = dzisiaj): ").strip()
    if user_date:
        try:
            date = datetime.strptime(user_date, "%Y-%m-%d").strftime("%Y-%m-%d")
        except ValueError:
            print("⚠️ Nieprawidłowy format daty! Użyto dzisiejszej daty.")
            date = datetime.today().strftime("%Y-%m-%d")
    else:
        date = datetime.today().strftime("%Y-%m-%d")
    print(f"📆 Użyta data badania: {date}")
    return date



In [3]:
def ocr_image_to_text(image_path, tesseract_cmd=r"C:\Program Files\Tesseract-OCR\tesseract.exe"):
    pytesseract.pytesseract.tesseract_cmd = tesseract_cmd
    text = pytesseract.image_to_string(Image.open(image_path), lang="pol")
    clean_text = re.sub(r'[\x00-\x08\x0B-\x0C\x0E-\x1F]', '', text)
    return clean_text


In [4]:
def parse_ocr_results(clean_text):
    rows = []
    for line in clean_text.splitlines():
        if not line.strip():
            continue
        match = re.match(r'(.+?)\s+([\d,\.]+)\s+([^\d]+)\s+(.+)', line)
        if match:
            badanie, wynik, jedn, zakres = match.groups()
            if re.search(r'\d', wynik) and re.search(r'\d', zakres):
                rows.append([badanie.strip(), jedn.strip(), zakres.strip(), wynik.strip()])
    df = pd.DataFrame(rows, columns=["Badanie", "Jedn.", "Zakres referencyjny", "Wynik"])
    return df


In [15]:
def enrich_with_norms(df, date):
    # Min/Max
    df[['Min','Max']] = df['Zakres referencyjny'].apply(
        lambda x: pd.Series(
            tuple(map(float, re.findall(r"[\d\.]+", x.replace(',','.'))[:2])) 
            if len(re.findall(r"[\d\.]+", x.replace(',','.'))) >= 2 
            else (None, None)
        )
    )

    # Status
    def check_status(val, min_val, max_val):
        try:
            v = float(val.replace(',', '.'))
            if pd.isna(min_val) or pd.isna(max_val): return "brak danych"
            if v < min_val: return "poniżej normy"
            if v > max_val: return "powyżej normy"
            return "w normie"
        except:
            return "brak danych"

    df[f"Wynik {date}"] = df["Wynik"]
    df[f"Status {date}"] = df.apply(lambda r: check_status(r["Wynik"], r["Min"], r["Max"]), axis=1)
    df = df.drop(columns=["Wynik"])
    return df


In [5]:
def merge_with_existing(df_new, excel_path, date):
    wynik_col = f"Wynik {date}"
    status_col = f"Status {date}"

    if os.path.exists(excel_path):
        df_existing = pd.read_excel(excel_path)

        # konwersja typów dla spójności
        for col in ["Min", "Max"]:
            if col in df_existing.columns:
                df_existing[col] = pd.to_numeric(df_existing[col], errors="coerce")
            if col in df_new.columns:
                df_new[col] = pd.to_numeric(df_new[col], errors="coerce")

        # upewnij się, że kolumny istnieją
        if wynik_col not in df_existing.columns:
            df_existing[wynik_col] = None
        if status_col not in df_existing.columns:
            df_existing[status_col] = None

        # scalanie po kluczowych kolumnach
        for _, row in df_new.iterrows():
            mask = (
                (df_existing["Badanie"] == row["Badanie"]) &
                (df_existing["Jedn."] == row["Jedn."]) &
                (df_existing["Zakres referencyjny"] == row["Zakres referencyjny"])
            )
            if mask.any():
                # jeśli badanie istnieje → uaktualnij wynik i status
                df_existing.loc[mask, wynik_col] = row[wynik_col]
                df_existing.loc[mask, status_col] = row[status_col]
            else:
                # jeśli nowe badanie → dodaj nowy wiersz
                new_row = {
                    "Badanie": row["Badanie"],
                    "Jedn.": row["Jedn."],
                    "Zakres referencyjny": row["Zakres referencyjny"],
                    "Min": row["Min"],
                    "Max": row["Max"],
                    wynik_col: row[wynik_col],
                    status_col: row[status_col]
                }
                df_existing = pd.concat([df_existing, pd.DataFrame([new_row])], ignore_index=True)

        df_merged = df_existing

    else:
        df_merged = df_new

    return df_merged


In [17]:
def color_status_columns(excel_path):
    wb = load_workbook(excel_path)
    ws = wb.active
    status_colors = {
        "w normie": "C6EFCE",
        "powyżej normy": "FFC7CE",
        "poniżej normy": "FFEB9C",
        "brak danych": "FFFFFF"
    }

    for col in range(1, ws.max_column + 1):
        header = ws.cell(row=1, column=col).value
        if header and header.startswith("Status"):
            for row in range(2, ws.max_row + 1):
                val = ws.cell(row=row, column=col).value
                ws.cell(row=row, column=col).fill = PatternFill(
                    start_color=status_colors.get(val, "FFFFFF"),
                    end_color=status_colors.get(val, "FFFFFF"),
                    fill_type="solid"
                )

    wb.save(excel_path)


In [21]:
def run_ocr_full2(image_path, excel_path, tesseract_cmd=r"C:\Program Files\Tesseract-OCR\tesseract.exe"):
    date = get_exam_date()
    clean_text = ocr_image_to_text(image_path, tesseract_cmd)
    df_raw = parse_ocr_results(clean_text)
    if df_raw.empty:
        print("⚠️ Nie znaleziono wyników.")
        return
    df_enriched = enrich_with_norms(df_raw, date)
    df_final = merge_with_existing(df_enriched, excel_path, date)
    df_final.to_excel(excel_path, index=False)
    color_status_columns(excel_path)
    print(f"\n✅ Zapisano wyniki do: {excel_path}")


In [26]:
image_path = "test3.png"
excel_path = "wyniki_badan.xlsx"
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

run_ocr_full2(image_path, excel_path)


📆 Użyta data badania: 2021-11-11

✅ Zapisano wyniki do: wyniki_badan.xlsx


In [27]:
def analyze_results(df, date):
    """Dodaje kolumnę Wynik {data} bez kolumny Status."""
    wynik_col = f"Wynik {date}"
    df.rename(columns={"Wynik": wynik_col}, inplace=True)
    return df

In [48]:

def merge_with_existing2(df_new, excel_path, date):
    """Scalanie wyników po nazwie badania, jednostce i zakresie referencyjnym."""
    wynik_col = f"Wynik {date}"

    if os.path.exists(excel_path):
        df_existing = pd.read_excel(excel_path)

        # Konwersja Min/Max do float dla bezpieczeństwa
        for col in ["Min", "Max"]:
            if col in df_existing.columns:
                df_existing[col] = pd.to_numeric(df_existing[col], errors="coerce")
            if col in df_new.columns:
                df_new[col] = pd.to_numeric(df_new[col], errors="coerce")

        # Upewnij się, że kolumna istnieje
        if wynik_col not in df_existing.columns:
            df_existing[wynik_col] = None

        # Aktualizuj istniejące lub dodaj nowe badania
        for _, row in df_new.iterrows():
            mask = (
                (df_existing["Badanie"] == row["Badanie"]) &
                (df_existing["Jedn."] == row["Jedn."]) &
                (df_existing["Zakres referencyjny"] == row["Zakres referencyjny"])
            )
            if mask.any():
                df_existing.loc[mask, wynik_col] = row[wynik_col]
            else:
                new_row = {
                    "Badanie": row["Badanie"],
                    "Jedn.": row["Jedn."],
                    "Zakres referencyjny": row["Zakres referencyjny"],
                    "Min": row.get("Min", np.nan),
                    "Max": row.get("Max", np.nan),
                    wynik_col: row[wynik_col]
                }
                df_existing = pd.concat([df_existing, pd.DataFrame([new_row])], ignore_index=True)

        df_merged = df_existing
    else:
        df_merged = df_new

    return df_merged

In [55]:
from openpyxl import load_workbook
from openpyxl.styles import PatternFill

def apply_color_formatting(excel_path):
    wb = load_workbook(excel_path)
    ws = wb.active

    headers = [cell.value for cell in ws[1]]
    wynik_cols = [i + 1 for i, h in enumerate(headers) if h.startswith("Wynik ")]
    min_idx = headers.index("Min") + 1 if "Min" in headers else None
    max_idx = headers.index("Max") + 1 if "Max" in headers else None

    red_fill = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")   # poza normą
    green_fill = PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid") # w normie

    for row in ws.iter_rows(min_row=2):
        # pobranie Min/Max
        try:
            min_val = float(str(row[min_idx-1].value).replace(",", ".")) if min_idx and row[min_idx-1].value not in [None, ""] else None
        except:
            min_val = None
        try:
            max_val = float(str(row[max_idx-1].value).replace(",", ".")) if max_idx and row[max_idx-1].value not in [None, ""] else None
        except:
            max_val = None

        # kolorowanie wyników
        for col_idx in wynik_cols:
            cell = row[col_idx-1]
            try:
                val = float(str(cell.value).replace(",", "."))
            except:
                continue  # pomiń nieprawidłowe wartości
            if (min_val is not None and val < min_val) or (max_val is not None and val > max_val):
                cell.fill = red_fill
            else:
                cell.fill = green_fill

    wb.save(excel_path)


In [68]:
def run_ocr_full(image_path, excel_path, tesseract_cmd=r"C:\Program Files\Tesseract-OCR\tesseract.exe"):
    # Pobranie daty badania
    date = get_exam_date()

    # OCR
    pytesseract.pytesseract.tesseract_cmd = tesseract_cmd
    text = pytesseract.image_to_string(Image.open(image_path), lang="pol")
    
    # Parsowanie do DataFrame
    df_new = parse_ocr_results(text)
    if df_new.empty:
        print("⚠️ Nie znaleziono wyników.")
        return

    # Analiza + Min/Max + Wynik {data}
    df_analyzed = enrich_with_norms(df_new, date)

    # Scalanie z istniejącym plikiem
    df_final = merge_with_existing2(df_analyzed, excel_path, date)

    # Zapis i kolorowanie
    try:
        df_final.drop(columns=[f"Status {date}"], inplace=True)
    except:
        pass
    df_final.to_excel(excel_path, index=False)
    apply_color_formatting(excel_path)
    
    print(f"✅ Wyniki zapisane i pokolorowane w pliku: {excel_path}")


In [69]:
image_path1 = "test3.png"
excel_path1 = "wyniki.xlsx"
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

run_ocr_full(image_path1, excel_path1)

📆 Użyta data badania: 2021-12-11
✅ Wyniki zapisane i pokolorowane w pliku: wyniki.xlsx


In [72]:
import matplotlib.pyplot as plt

wyniki = pd.read_excel(excel_path1)
wyniki.head()

,Badanie,Jedn.,Zakres referencyjny,Min,Max,Wynik 2025-10-31,Wynik 2021-12-11
0,Monocyty,%*,"2,0 - 10,0",2.0,10.00,"9,7","9,7"
1,Eozynofile,%*,"1,0-6,0",1.0,6.00,"1,5","1,5"
2,Bazofile,%,"0,0-2,0",0.0,2.00,11,11
3,NRBC%,%,"0,00 - 0,01",0.0,0.01,"0,00","0,00"
4,NRBC$,tys/ul*,"0,00 - 0,03",0.0,0.03,"0,00","0,00"


In [ ]:
def generate_report_with_charts(excel_path, output_txt="raport.txt", charts_dir="charts"):
    # Wczytanie danych
    df = pd.read_excel(excel_path)

    # Znajdź kolumny z wynikami
    wynik_cols = [col for col in df.columns if col.startswith("Wynik")]
    if not wynik_cols:
        print("⚠️ Nie znaleziono kolumn z wynikami.")
        return


    # Otwórz plik txt
    with open(output_txt, "w", encoding="utf-8") as f:
        f.write("=== RAPORT WYNIKÓW BADAŃ ===\n\n")
        for idx, row in df.iterrows():
            f.write(f"Nazwa badania: {row['Badanie']}\n")
            f.write(f"Zakres referencyjny: {row.get('Zakres referencyjny', 'brak')}\n")

            # Lista do wykresu
            dates = []
            values = []

            min_val = row.get("Min")
            max_val = row.get("Max")

            for col in wynik_cols:
                wynik = row[col]
                date = col.replace("Wynik ", "")
                dates.append(date)
                try:
                    val = float(str(wynik).replace(",", "."))
                    values.append(val)
                except:
                    values.append(None)
                # Status
                if pd.isna(wynik) or min_val is None or max_val is None:
                    status = "brak danych"
                else:
                    try:
                        if val < min_val:
                            status = "poniżej normy"
                        elif val > max_val:
                            status = "powyżej normy"
                        else:
                            status = "w normie"
                    except:
                        status = "brak danych"

                f.write(f"  {col}: {wynik} → {status}\n")
            f.write("\n")


    print(f"Raport zapisany do: {output_txt}")


In [74]:
generate_report_with_charts(excel_path1)

✅ Raport zapisany do: raport.txt
✅ Wykresy zapisane w folderze: charts
